In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch
import numpy as np
from typing import List
from diffusers import StableDiffusionPipeline
from IPython.display import display

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16,
    safety_checker=None
).to(device)

model.enable_attention_slicing()
model.enable_vae_slicing()

print("✓ Model loaded")


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its result

✓ Model loaded


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [4]:
@torch.no_grad()
def generate(prompt, negative_prompt=None, seed=None):

    generator = None
    if seed is not None:
        generator = torch.Generator(device=device).manual_seed(seed)

    image = model(
        prompt=prompt,
        negative_prompt=negative_prompt,   # ← added
        num_inference_steps=50,
        guidance_scale=7.5,
        generator=generator
    ).images[0]

    return image

In [5]:
import os
import torch
from pathlib import Path
from IPython.display import display
from diffusers import StableDiffusionXLImg2ImgPipeline

# ===============================
# ENTRY POINT
# ===============================
if __name__ == "__main__":

    # List of prompts to evaluate for compositional logic
    prompts = [
            "A ball roughly next to a basket"
    ]
    
    # Note: Ensure get_token_indices extracts the correct nouns for each prompt
    # Here we define the tracking words per prompt index
    tracking_words = [
        ["ball", "basket"]
    ]

    negative_prompt = (
        "same size object, equal size, merged objects, "
        "single subject, blurry, low quality, cartoon, illustration, "
        "3d render, painting, text, watermark"
    )

    # Generate seeds from 42 to 142 inclusive (101 total)
    seeds = [142,742,842]


    output_root = Path("./aesthetic_Batch")
    output_root.mkdir(exist_ok=True)

    for p_idx, prompt_text in enumerate(prompts):
        # Create a subfolder for each specific prompt to avoid clutter
        prompt_folder_name = prompt_text[:30].replace(" ", "_")
        prompt_path = output_root / f"prompt_{p_idx}_{prompt_folder_name}"
        prompt_path.mkdir(exist_ok=True)
        
        words = tracking_words[p_idx]

        print(f"\n🚀 Processing Prompt {p_idx+1}/{len(prompts)}: {prompt_text}")

        for seed_value in seeds:
            print(f"Generating Seed: {seed_value}...", end="\r")

            # ===============================
            # Stage 1 — FUZZY GENERATION
            # ===============================
            image =  generate(prompt=prompt_text,  negative_prompt=negative_prompt,seed=seed_value)

            if image is None:
                continue

            # Save Stage-1 image
            s1_filename = f"seed_{seed_value}_stage1.png"
            image.save(prompt_path / s1_filename)


    print(f"\n✨ Batch complete. All images saved in: {output_root.absolute()}")


🚀 Processing Prompt 1/1: A ball roughly next to a basket


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]


✨ Batch complete. All images saved in: /kaggle/working/aesthetic_Batch
